# งานตรวจในโรงงาน 4 แบบ ที่ไม่ได้ดูแค่สี
Workshop ก่อนหน้าแยกวัตถุด้วย**สี** ซึ่งใช้ได้กับงานบางส่วนเท่านั้น งานจริงส่วนใหญ่เป็นแบบนี้:

| งาน | เทคนิค | ใช้กับ |
|---|---|---|
| 1. ตรวจสิ่งแปลกปลอม / ชิ้นหาย | เทียบกับภาพอ้างอิง (Reference) | ถาดบรรจุ, แผงวงจร, ชิ้นงานในจิ๊ก |
| 2. ตรวจฉลากสินค้า | หาตำแหน่ง (Template matching) + เทียบกับฉลากต้นแบบ + OCR | ฉลาก, กล่อง, การพิมพ์วันที่ |
| 3. นับและแยกชนิดชิ้นงาน | Threshold + แยกชิ้นที่ติดกัน + รูปทรง | น็อต แหวน ชิ้นส่วนโลหะ |
| 4. วัดขนาด | หาขอบ + สอบเทียบ พิกเซล → มิลลิเมตร | ตรวจมิติ |

ทุกวิธีเป็น Traditional CV ใช้กับ**ชิ้นงานที่หน้าตาคงที่ ในจิ๊กที่ยึดตำแหน่ง และแสงที่คุมได้** ซึ่งคือสภาพของสถานีตรวจส่วนใหญ่

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath("../scripts"))
import cv2
import numpy as np
from vision_camera import show, img_path, put_label
IND = img_path("industrial")                      # โฟลเดอร์ภาพตัวอย่างงานอุตสาหกรรม
def load(name):
    return cv2.imread(os.path.join(IND, name))
print("พร้อม  ภาพตัวอย่าง:", len(os.listdir(IND)), "ไฟล์")

## 1. ตรวจสิ่งแปลกปลอมและชิ้นหาย: เทียบกับภาพอ้างอิง
**หลักการ:** ถ่ายภาพถาดที่ถูกต้องเก็บไว้ 1 ภาพ (Reference / Golden sample) แล้วทุกถาดใหม่ให้**ลบภาพ**กัน ส่วนที่ต่างคือสิ่งที่ไม่ควรมี หรือสิ่งที่หายไป
ใช้ได้กับสิ่งแปลกปลอม**ทุกสี ทุกรูปร่าง** แม้ไม่เคยเห็นมาก่อน เพราะไม่ได้หาสิ่งแปลกปลอม แต่หา "สิ่งที่ไม่เหมือนภาพอ้างอิง"

**เงื่อนไข:** กล้องและชิ้นงานต้องอยู่ตำแหน่งเดิมทุกครั้ง (จิ๊กยึด) และแสงคงที่

In [ ]:
ref = load("tray_reference.png")
show(ref, titles=["ภาพอ้างอิง: ถาดที่ถูกต้อง"], size=5)

def compare_to_reference(img, ref, diff_threshold=20, min_area=120):
    """คืน (ภาพผล, รายการความต่าง)  แต่ละความต่างบอกว่า 'เกินมา' หรือ 'หายไป'"""
    g1 = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    g0 = cv2.GaussianBlur(cv2.cvtColor(ref, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    diff = cv2.absdiff(g1, g0)                                  # ส่วนที่ต่างจากภาพอ้างอิง
    _, mask = cv2.threshold(diff, diff_threshold, 255, cv2.THRESH_BINARY)
    mask = cv2.dilate(mask, np.ones((5, 5), np.uint8))          # รวมพิกเซลที่ต่างให้เป็นชิ้นเดียวกัน (เส้นผมบางก็ยังอยู่)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = img.copy(); findings = []
    for c in contours:
        if cv2.contourArea(c) < min_area:
            continue
        x, y, w, h = cv2.boundingRect(c)
        roi = (slice(y, y + h), slice(x, x + w))
        # ภาพอ้างอิงตรงนี้เคยมีชิ้นงาน (สว่าง) = หายไป  ถ้าเดิมเป็นพื้นถาด = มีของแปลกปลอมเพิ่มมา
        kind = "หายไป" if g0[roi].mean() > 120 else "สิ่งแปลกปลอม"
        color = (0, 165, 255) if kind == "หายไป" else (0, 0, 255)
        cv2.rectangle(out, (x, y), (x + w, y + h), color, 3)
        findings.append((kind, int(cv2.contourArea(c)), (x, y, w, h)))
    return out, findings, diff

for name in ["tray_01_clean.png", "tray_02_screw.png", "tray_03_hair.png",
             "tray_04_missing.png", "tray_05_plastic.png"]:
    out, findings, diff = compare_to_reference(load(name), ref)
    verdict = "OK" if not findings else "NG: " + ", ".join(f"{k} ({a} px)" for k, a, _ in findings)
    print(f"{name:22s} -> {verdict}")
    show(out, diff, titles=[name + "  " + ("OK" if not findings else "NG"), "ความต่างจากภาพอ้างอิง"], size=4)

**สังเกต:** เส้นผมบางๆ ก็จับได้ เพราะเทียบทีละพิกเซล  แต่วิธีนี้**ไวต่อการขยับ** ถ้าถาดเลื่อนไป 5 พิกเซล ทุกชิ้นจะกลายเป็น "ต่าง" หมด
**ลองเอง:** เปลี่ยน `diff_threshold` เป็น 8 และ 60 ดูว่าจุดรบกวนและเส้นผมเปลี่ยนอย่างไร

## 2. ตรวจฉลากสินค้า: ตรงกับที่ออกแบบไว้ไหม
ตรวจ 3 ชั้น เหมือนสถานีตรวจฉลากจริง
1. **มีฉลากไหม อยู่ตรงไหน** หาด้วย Template matching จากฉลากต้นแบบ
2. **การพิมพ์ครบไหม** ตัดฉลากออกมาเทียบกับต้นแบบ (วิธีเดียวกับข้อ 1)
3. **ข้อความถูกไหม** อ่านด้วย OCR แล้วเทียบกับที่ควรเป็น (เช่น ชื่อสินค้า และต้องมี EXP)

In [ ]:
import pytesseract, shutil, difflib
WIN = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
if shutil.which("tesseract") is None and os.path.exists(WIN):
    pytesseract.pytesseract.tesseract_cmd = WIN

golden = load("label_golden.png")
EXPECT_PRODUCT = "PRODUCT A"
show(golden, titles=["ฉลากต้นแบบ (Golden)"], size=5)

def check_label(img, golden, match_min=0.6, diff_max=0.01, name_min=0.85):
    gi = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY); gg = cv2.cvtColor(golden, cv2.COLOR_BGR2GRAY)
    # 1) หาตำแหน่งฉลาก
    res = cv2.matchTemplate(gi, gg, cv2.TM_CCOEFF_NORMED)
    _, score, _, (x, y) = cv2.minMaxLoc(res)
    h, w = gg.shape
    out = img.copy(); reasons = []
    if score < match_min:
        cv2.putText(out, f"label not found ({score:.2f})", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        return out, ["ไม่พบฉลาก / ฉลากเอียง-เลื่อน (match " + f"{score:.2f})"]
    cv2.rectangle(out, (x, y), (x + w, y + h), (0, 200, 0), 3)
    crop = gi[y:y + h, x:x + w]
    # 2) เทียบการพิมพ์กับต้นแบบ
    diff = cv2.absdiff(crop, gg)
    _, m = cv2.threshold(diff, 60, 255, cv2.THRESH_BINARY)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    diff_ratio = m.mean() / 255
    if diff_ratio > diff_max:
        reasons.append(f"การพิมพ์ต่างจากต้นแบบ {diff_ratio*100:.1f}% ของพื้นที่")
        cs, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for c in cs:
            if cv2.contourArea(c) > 80:
                bx, by, bw, bh = cv2.boundingRect(c)
                cv2.rectangle(out, (x + bx, y + by), (x + bx + bw, y + by + bh), (0, 0, 255), 2)
    # 3) อ่านข้อความ
    text = pytesseract.image_to_string(crop, config="--psm 6").upper()
    first = text.split(chr(10))[0] if text else ""
    # OCR อ่านตัวอักษรเพี้ยนได้บ้าง (เช่น D เป็น V) จึงใช้ "ความคล้าย" แทนการเทียบตรงๆ
    L = len(EXPECT_PRODUCT)
    name_sim = max((difflib.SequenceMatcher(None, EXPECT_PRODUCT, first[i:i + L]).ratio()
                    for i in range(max(1, len(first) - L + 1))), default=0)
    if name_sim < name_min:
        reasons.append(f"ชื่อสินค้าไม่ตรง (อ่านได้ '{first.strip()}' คล้าย {name_sim:.0%})")
    if "EXP" not in text:
        reasons.append("ไม่มีวันหมดอายุ")
    return out, reasons

for name in ["label_01_good.png", "label_02_no_exp.png", "label_03_wrong_product.png",
             "label_04_tilted.png", "label_05_smudge.png"]:
    out, reasons = check_label(load(name), golden)
    print(f"{name:26s} -> " + ("OK" if not reasons else "NG: " + " | ".join(reasons)))
    show(out, titles=[name + "  " + ("OK" if not reasons else "NG")], size=5)

**สังเกต:**
- ฉลาก**เอียง 7 องศา** ระบบบอก "ไม่พบฉลาก" เพราะ Template matching ไม่ทนต่อการหมุน  ในงานจริงแก้ได้ 2 ทาง: ใช้จิ๊กยึดชิ้นงานให้ตรง (ถูกและเสถียรที่สุด) หรือใช้วิธีหาฉลากที่ทนการหมุน (เช่น feature matching) ซึ่งซับซ้อนขึ้น
- รอยเปื้อนจับได้จากการเทียบกับต้นแบบ แม้ OCR ยังอ่านข้อความได้
**ลองเอง:** เปลี่ยน `EXPECT_PRODUCT` เป็น "PRODUCT B" แล้วดูว่าภาพไหนกลายเป็น OK  และลองลด `diff_max` เหลือ 0.001 ดูว่าฉลากที่พิมพ์ชื่อผิดถูกจับได้จากการเทียบภาพด้วยไหม

## 3. นับและแยกชนิดชิ้นงานโลหะ โดยไม่ใช้สี
แหวนกับน็อตสีเดียวกัน แยกด้วยสีไม่ได้  ใช้ **ความสว่าง + รูปทรง** แทน
- Threshold แบบ Otsu แยกโลหะ (สว่าง) ออกจากพื้น (มืด)
- ชิ้นที่**วางติดกัน** แยกด้วย Distance transform + Watershed (ไม่งั้นจะนับเป็นชิ้นเดียว)
- แยกชนิดด้วย**ขนาดรู**: แหวนรูใหญ่ น็อตรูเล็ก (ลักษณะที่วัดได้แม้ชิ้นงานติดกัน)

In [ ]:
def count_parts(img, peak_frac=0.5, hole_min=280):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, bw = cv2.threshold(cv2.GaussianBlur(gray, (5, 5), 0), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cs, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled = np.zeros_like(bw); cv2.drawContours(filled, cs, -1, 255, -1)       # ชิ้นงานแบบอุดรู
    holes = cv2.bitwise_and(filled, cv2.bitwise_not(bw))                          # เฉพาะรูตรงกลาง

    # แยกชิ้นที่ติดกัน: Distance transform หา "ใจกลาง" ของแต่ละชิ้น แล้ว Watershed แบ่งเขต
    dist = cv2.distanceTransform(filled, cv2.DIST_L2, 5)
    _, peaks = cv2.threshold(dist, peak_frac * dist.max(), 255, 0)
    peaks = peaks.astype(np.uint8)
    sure_bg = cv2.dilate(filled, np.ones((9, 9), np.uint8))
    n_markers, markers = cv2.connectedComponents(peaks)
    markers = markers + 1
    markers[(sure_bg == 255) & (peaks == 0)] = 0        # บริเวณที่ยังไม่รู้ว่าเป็นของชิ้นไหน ให้ Watershed ตัดสิน
    markers = cv2.watershed(img, markers)

    out = img.copy(); counts = {"washer": 0, "nut": 0}
    for label_id in range(2, n_markers + 1):
        region = markers == label_id
        if region.sum() < 300:
            continue
        hole_area = int((holes[region] > 0).sum())
        kind = "washer" if hole_area > hole_min else "nut"
        counts[kind] += 1
        ys, xs = np.where(region)
        color = (0, 255, 0) if kind == "washer" else (0, 200, 255)
        out[markers == -1] = (0, 0, 255)                                           # เส้นแบ่งของ Watershed
        put_label(out, f"{kind[0].upper()} hole {hole_area}", org=(int(xs.min()), int(ys.min()) - 4), scale=0.45, color=color)
    return out, counts, bw

for name in ["parts_01.png", "parts_02.png"]:
    out, counts, bw = count_parts(load(name))
    print(f"{name}: แหวน {counts['washer']}  น็อต {counts['nut']}")
    show(bw, out, titles=["หลัง Threshold (Otsu)", f"แหวน {counts['washer']}  น็อต {counts['nut']}  (เส้นแดง = รอยตัดของ Watershed)"], size=5)

**สังเกต:** ชิ้นที่วางติดกัน ถ้านับจาก Threshold ตรงๆ จะได้ก้อนเดียว Watershed ช่วยแยกออกจากกัน แต่ถ้าซ้อนทับกันมากก็ยังแยกไม่ได้ ในไลน์จริงจึงมักมีกลไกเกลี่ยชิ้นงานให้กระจายก่อนเข้ากล้อง
**ลองเอง:** เปลี่ยน `peak_frac` เป็น 0.3 และ 0.8 ดูว่าการแยกชิ้นที่ติดกันเปลี่ยนไปอย่างไร (ต่ำไป = ชิ้นติดกันไม่แยก  สูงไป = ชิ้นเดียวถูกตัดเป็นหลายชิ้น)

## 4. วัดขนาดเป็นมิลลิเมตร
กล้องเห็นเป็น**พิกเซล** ต้องสอบเทียบ (Calibrate) ด้วยวัตถุที่รู้ขนาดจริง เช่น เหรียญ หรือแผ่นสอบเทียบ แล้วได้ค่า **พิกเซลต่อมิลลิเมตร** จากนั้นวัดชิ้นงานอื่นในภาพเดียวกันได้
ใช้ได้เมื่อ: ชิ้นงานแบนและอยู่ระยะเดียวกับวัตถุสอบเทียบ (ระยะต่างกัน = ขนาดต่อพิกเซลต่างกัน)

In [ ]:
img = load("measure_01.png")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
_, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)   # วัตถุมืดบนพื้นสว่าง
cs, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cs = sorted(cs, key=lambda c: cv2.boundingRect(c)[0])                           # ซ้ายไปขวา: เหรียญก่อน

COIN_MM = 20.0
(cx, cy), r = cv2.minEnclosingCircle(cs[0])
px_per_mm = (2 * r) / COIN_MM
print(f"เหรียญกว้าง {2*r:.0f} px = {COIN_MM} mm  ->  {px_per_mm:.2f} px/mm")

out = img.copy()
cv2.circle(out, (int(cx), int(cy)), int(r), (0, 200, 0), 2)
for c in cs[1:]:
    rect = cv2.minAreaRect(c)                       # กรอบที่หมุนตามชิ้นงาน
    (w, h) = rect[1]
    box = cv2.boxPoints(rect).astype(int)
    cv2.drawContours(out, [box], 0, (0, 0, 255), 2)
    W, H = sorted([w / px_per_mm, h / px_per_mm], reverse=True)
    put_label(out, f"{W:.1f} x {H:.1f} mm", org=(box[:, 0].min(), box[:, 1].min() - 10), scale=0.7)
    print(f"ชิ้นงาน: {W:.2f} x {H:.2f} mm   (ความละเอียด 1 px = {1/px_per_mm:.2f} mm)")
show(out, titles=["สอบเทียบด้วยเหรียญ 20 mm แล้ววัดชิ้นงาน"], size=6)

**สังเกต:** ค่าที่วัดได้ละเอียดแค่ประมาณ 1 พิกเซล (ที่นี่ 0.1 มม.) ถ้าต้องการวัดละเอียดกว่านี้ต้องเพิ่มพิกเซลต่อมิลลิเมตร คือซูมเข้า (FOV เล็กลง) หรือใช้กล้องละเอียดขึ้น ตรงกับการคำนวณในวันแรก และงานวัดที่แม่นจริงมักใช้ **เลนส์ Telecentric** (ไม่มีการย่อ-ขยายตามระยะ) กับไฟส่องจากด้านหลัง (Backlight) ให้ขอบคม

## สรุป: เลือกวิธีตามงาน

| ถ้าโจทย์คือ | ใช้ | ต้องมี |
|---|---|---|
| มีอะไรแปลกปลอม / ขาดไหม | เทียบภาพอ้างอิง | จิ๊กยึดตำแหน่ง แสงคงที่ |
| พิมพ์/ฉลากถูกไหม | Template + เทียบต้นแบบ + OCR | ฉลากต้นแบบ, ตำแหน่งไม่เอียง |
| มีกี่ชิ้น ชนิดไหน | Threshold + Watershed + รูปทรง | ชิ้นงานไม่ซ้อนกัน |
| ขนาดเท่าไร | สอบเทียบ px/mm | วัตถุสอบเทียบ ระยะคงที่ |
| หน้าตาไม่คงที่ ตำหนิหลากหลาย | **AI (Classification / Detection)** | ภาพตัวอย่างจำนวนมาก |

ทั้ง 4 วิธีใช้ฟังก์ชันไม่กี่ตัวของ OpenCV และรันได้ในหลักสิบมิลลิวินาที ซึ่งเพียงพอกับไลน์ผลิตส่วนใหญ่